# CCM-109 - Tópicos especiais de IA - Deep Learning

## Pipeline de detecção de deepfake

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jtlimo/ccm-109/blob/main/train.ipynb)

---

### Índice

1. [Configuração](#1-configuração)
2. [Dataset FF++](#2-dataset-faceforensics)
3. [Funções auxiliares](#3-funções-auxiliares)
4. [Treino](#4-treino)

## 1. Configuração

In [23]:
# @title Instala dependências

%pip install -q tensorflow keras-hub numpy opencv-python matplotlib scikit-learn mtcnn ipywidgets tqdm

In [24]:
# @title Imports

import os
import glob
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score
import model as m
import preprocess as pi
import keras_hub

In [25]:
# @title Configurações

IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 5
LR = 1e-4
FEATURE_DIM = 768
DATASET_ROOT = "/content/drive/MyDrive/dataset/FaceForensics"
DATASET_LOCAL = "/content/dataset_local"
FEATURES_SAVE_DIR = "/content/drive/MyDrive/dataset/FF_Features_Cache"
FEATURES_CACHE = "/content/drive/MyDrive/dataset/siglip2_features_keras.npz"

import tensorflow as tf
from tensorflow import keras

AUTOTUNE = tf.data.AUTOTUNE

print(f"GPUs visíveis: {tf.config.list_physical_devices('GPU')}")
print(f'TensorFlow: {tf.__version__}')

GPUs visíveis: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
TensorFlow: 2.20.0


## 2. Dataset FaceForensics++

No arquivo **face_extraction.ipynb** foi realizada a extração dos rostos frame por frame nos vídeos do dataset. Para melhorar a otimização, foram utilizados os seguintes filtros na extração:

- SKIP_FRAMES = 9     
_0 = todos; 9 = pula 9 (processa 1 a cada 10)_

- MAX_FRAMES_PER_VIDEO = 30          
_None = todos; ex: 100 = máx 100 frames que contenham rostos são salvos_

- MAX_READ_LIMIT = 1000                 
_None = todos; ex: 1000 = máx 1000 frames lidos por vídeo_

- MAX_VIDEOS = None                 
_None = todos; ex: 10 = máx 10 vídeos processados_


| Índice | Classe
|---|---
| 0 | real
| 1 | fake

Há também o pré-processamento para o siglip2 que é o que será usado no modelo, as imagens são salvas em 224x224 (float 32)

## 3. Funções auxiliares

In [26]:
#@title Verifica se o backbone existe localmente

if os.path.exists("siglip2_base_patch16_224_original_backbone.keras"):
   print("Carregando backbone do SigLIP2 a partir do arquivo salvo...")
   try:
        backbone = keras.models.load_model("siglip2_base_patch16_224_original_backbone.keras")
   except Exception as e:
        print(f"Erro ao carregar o backbone localmente: {e}")
        print("Recarregando o backbone do SigLIP2 a partir do preset...")
        backbone = keras_hub.models.SigLIPBackbone.from_preset("siglip2_base_patch16_224")
        backbone.trainable = False
else:
  print("Baixando o backbone SigLIP2 do KerasHub...")
  backbone = keras_hub.models.SigLIPBackbone.from_preset("siglip2_base_patch16_224")
  backbone.trainable = False
  backbone.save("siglip2_base_patch16_224_original_backbone.keras")
  print("Salvo localmente")

Carregando backbone do SigLIP2 a partir do arquivo salvo...


In [27]:
# @title Carrega Siglip2 sem a cabeça classificadora
def extract_features(file_paths, backbone_model, batch_size=32):
    features = []

    for i in range(0, len(file_paths), batch_size):
        batch_files = file_paths[i:i+batch_size]
        batch_imgs = np.array([np.load(f) for f in batch_files])

        batch_feats = backbone_model.get_vision_embeddings(batch_imgs)
        features.append(batch_feats.numpy())

    return np.concatenate(features, axis=0)


In [28]:
# @title Carrega imagens pré-processadas para o Siglip2

def load_dataset_metadata(dataset_dir):
    file_paths, labels, video_ids = [], [], []
    search_path = os.path.join(dataset_dir, "**", "*_siglip2.npy")

    all_files = glob.glob(search_path, recursive=True)

    for filepath in all_files:
        path_parts = filepath.split(os.sep)
        full_path_str = filepath.lower()


        label = 1 if any(kw in full_path_str for kw in ["fake", "manipulated", "deepfake"]) else 0

        raw_video_id = path_parts[-2]
        video_id = f"{'fake' if label == 1 else 'real'}_{raw_video_id}"

        file_paths.append(filepath)
        labels.append(label)
        video_ids.append(video_id)

    return np.array(file_paths), np.array(labels), np.array(video_ids)



In [ ]:
#@title Processamento dos .zip
import shutil

search_path = os.path.join(DATASET_ROOT, "**", "*.zip")
zip_files = sorted(glob.glob(search_path, recursive=True))
print(f"📦 Encontrados {len(zip_files)} arquivos .zip para processar.\n")

for idx, zip_path in enumerate(zip_files, 1):
    zip_name = os.path.basename(zip_path)
    output_npz_name = zip_name.replace(".zip", "_features.npz")
    output_npz_path = os.path.join(FEATURES_SAVE_DIR, output_npz_name)

    if os.path.exists(output_npz_path):
        print(f"⏩ [{idx}/{len(zip_files)}] {zip_name} já foi processado. Pulando...")
        continue

    print("=" * 70)
    print(f"🚀 [{idx}/{len(zip_files)}] Processando: {zip_name}")
    print("=" * 70)

    if os.path.exists(DATASET_LOCAL):
        shutil.rmtree(DATASET_LOCAL)
    os.makedirs(DATASET_LOCAL, exist_ok=True)

    temp_zip = "/content/temp_chunk.zip"
    print(" 1. Copiando .zip para o SSD da VM...")
    !cp "{zip_path}" "{temp_zip}"

    print(" 2. Descompactando arquivos...")
    !unzip -q "{temp_zip}" -d "{DATASET_LOCAL}"
    os.remove(temp_zip)

    file_paths, labels, video_ids = load_dataset_metadata(DATASET_LOCAL)
    print(f" 3. Imagens encontradas: {len(file_paths):,} | Reais (0): {np.sum(labels == 0)} | Fakes (1): {np.sum(labels == 1)}")

    if len(file_paths) == 0:
        print(" ⚠️ Nenhuma imagem .npy encontrada neste arquivo zip. Pulando...")
        continue

    print(" 4. Extraindo embeddings de visão (SigLIP2)...")
    features = extract_features(file_paths, backbone, batch_size=64)

    print(f" 💾 Salvando cache em: {output_npz_name}...")
    np.savez_compressed(
        output_npz_path,
        features=features,
        labels=labels,
        video_ids=video_ids
    )

    shutil.rmtree(DATASET_LOCAL)
    tf.keras.backend.clear_session()
    print(" ✅ Concluído e disco limpo!\n")

📦 Encontrados 8 arquivos .zip para processar.

🚀 [1/8] Processando: DeepFakeDetection.zip
 1. Copiando .zip para o SSD da VM...
 2. Descompactando arquivos...
/content/dataset_local/DeepFakeDetection/c23/videos/27_20__walking_and_outside_surprised__LM907ECA/frame_000161_face_002_siglip2.npy:  write error (disk full?).  Continue? (y/n/^C) 

In [ ]:
# @title Carregamento dos Embeddings e Divisão em Treino, Validação e Teste

npz_files = sorted(glob.glob(os.path.join(FEATURES_SAVE_DIR, "*_features.npz")))
print(f"📦 Encontrados {len(npz_files)} arquivos de features .npz no cache.")

if not npz_files:
    raise FileNotFoundError(f"Nenhum arquivo .npz encontrado em {FEATURES_SAVE_DIR}. Execute a Parte 1 primeiro.")

all_features, all_labels, all_vids = [], [], []

for npz_path in npz_files:
    data = np.load(npz_path)
    all_features.append(data["features"])
    all_labels.append(data["labels"])
    all_vids.append(data["video_ids"])

X = np.concatenate(all_features, axis=0)
y = np.concatenate(all_labels, axis=0)
vids = np.concatenate(all_vids, axis=0)

print(f"\n✅ Total de embeddings em RAM: {X.shape} ({X.nbytes / 1e6:.1f} MB)")
print(f"Distribuição total - Reais (0): {np.sum(y == 0):,} | Fakes (1): {np.sum(y == 1):,}")

gss_test = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_val_idx, test_idx = next(gss_test.split(X, y, groups=vids))

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=42)
train_sub_idx, val_sub_idx = next(gss_val.split(X[train_val_idx], y[train_val_idx], groups=vids[train_val_idx]))

real_train_idx = train_val_idx[train_sub_idx]
real_val_idx   = train_val_idx[val_sub_idx]

X_train, y_train = X[real_train_idx], y[real_train_idx]
X_val, y_val     = X[real_val_idx], y[real_val_idx]
X_test, y_test   = X[test_idx], y[test_idx]

print("\n" + "=" * 50)
print(f"Splits Finais:")
print(f"  • Treino:    {X_train.shape[0]:,} amostras")
print(f"  • Validação: {X_val.shape[0]:,} amostras")
print(f"  • Teste:     {X_test.shape[0]:,} amostras")
print("=" * 50)

## 4. Treino

In [ ]:
# @title Cria datasets de treino, validação e teste

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(len(X_train)).batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE).prefetch(AUTOTUNE)

In [ ]:
FEATURE_DIM = X_train.shape[1]  # Ex: 768
classifier = m.build_classifier(feature_dim=FEATURE_DIM, l2_reg=1e-4)

classifier.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

classifier.summary()

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_auc',
        patience=2,
        mode='max',
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        'model.{epoch:02d}-{val_auc:.4f}.keras',
        monitor='val_auc',
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

history = classifier.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    batch_size=64,
    callbacks=callbacks
)

print(f'\nAcurácia de validação (Fase 1): {history.history["val_accuracy"][-1]:.1%}')

In [ ]:
#@title Avaliação do modelo

val_preds = classifier.predict(X_val).flatten()
val_preds_binary = (val_preds > 0.5).astype(int)

print("\n--- Relatório de Classificação (Validação) ---")
print(classification_report(y_val, val_preds_binary, target_names=['Real', 'Fake']))
print(f"AUC-ROC Final: {roc_auc_score(y_val, val_preds):.4f}")

In [ ]:
# @title Avaliação final no conjunto de teste

oi = classifier.evaluate(test_ds, verbose=0)
print(f'Loss final no teste   : {oi:.4f}')
print(f'Acurácia final no teste: {oi:.1%}')

In [ ]:
# @title Plota métricas de treino e validação

acc_all     = history.history['accuracy']
val_acc_all = history.history['val_accuracy']

auc_all     = history.history['auc']
val_auc_all = history.history['val_auc']

loss_all     = history.history['loss']
val_loss_all = history.history['val_loss']


# Acurácia

plt.figure(figsize=(10, 5))
plt.plot(acc_all,     label='Treino',     linewidth=2)
plt.plot(val_acc_all, label='Validação',  linewidth=2)
plt.fill_betweenx([0, 1], 0, EPOCHS - 0.5,
                  alpha=0.05, color='blue')
plt.title('Acurácia — Transfer Learning no FF++', fontsize=13)
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.legend(loc='lower right')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()


# AUC

plt.figure(figsize=(10, 5))
plt.plot(auc_all,     label='Treino',     linewidth=2)
plt.plot(val_auc_all, label='Validação',  linewidth=2)
plt.fill_betweenx([0, 1], 0, EPOCHS - 0.5,
                  alpha=0.05, color='blue')
plt.title('AUC — Transfer Learning no FF++', fontsize=13)
plt.xlabel('Época')
plt.ylabel('AUC')
plt.legend(loc='lower right')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()


# Loss

plt.figure(figsize=(10, 5))
plt.plot(loss_all,     label='Treino',     linewidth=2)
plt.plot(val_loss_all, label='Validação',  linewidth=2)
plt.fill_betweenx([0, 1], 0, EPOCHS - 0.5,
                  alpha=0.05, color='blue')
plt.title('Perda — Transfer Learning no FF++', fontsize=13)
plt.xlabel('Época')
plt.ylabel('Perda')
plt.legend(loc='lower right')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()




In [ ]:
# #@title Descongela as últimas 4 camadas do vision encoder
# model.trainable = True

# freeze_until = len(backbone.vision_encoder.layers) - 4
# for layer in backbone.vision_encoder.layers[:freeze_until]:
#     layer.trainable = False

# trainable = sum(1 for l in model.layers if l.trainable)
# frozen    = sum(1 for l in model.layers if not l.trainable)
# print(f'Camadas treináveis na base: {trainable}')
# print(f'Camadas congeladas na base: {frozen}')

# trainable_ve = sum(1 for l in backbone.vision_encoder.layers if l.trainable)
# frozen_ve    = sum(1 for l in backbone.vision_encoder.layers if not l.trainable)
# print(f'Vision encoder — Treináveis: {trainable_ve}, Congeladas: {frozen_ve}')

# backbone.save("siglip2_base_patch16_224_finetune_backbone.keras")


In [ ]:
# @title Re-compila com taxa de aprendizado menor — Fase 2

# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
#     loss='binary_crossentropy',
#     metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
# )

# model.summary(show_trainable=True)

In [ ]:
# #@title Treinamento Fase 2 (5 epochs, LR=1e-5)

# EPOCHS_2 = 5

# history2 = model.fit(
#     train_ds,
#     epochs=EPOCHS + EPOCHS_2,
#     initial_epoch=EPOCHS,
#     validation_data=val_ds,
#     verbose=1,
#     callbacks=callbacks
# )

# print(f'\nAcurácia de validação (Fase 2): {history2.history["val_accuracy"][-1]:.1%}')

In [ ]:
# # @title Concatena históricos e plota
# acc_all     = history.history['accuracy']     + history2.history['accuracy']
# val_acc_all = history.history['val_accuracy'] + history2.history['val_accuracy']

# plt.figure(figsize=(10, 5))
# plt.plot(acc_all,     label='Treino',     linewidth=2)
# plt.plot(val_acc_all, label='Validação',  linewidth=2)
# plt.axvline(x=EPOCHS - 0.5, color='gray', linestyle='--', linewidth=1.5,
#             label='Início do fine-tuning')
# plt.fill_betweenx([0, 1], 0, EPOCHS - 0.5,
#                   alpha=0.05, color='blue',  label='Extração de características')
# plt.fill_betweenx([0, 1], EPOCHS - 0.5, len(acc_all),
#                   alpha=0.05, color='orange', label='Fine-tuning')
# plt.title('Acurácia — Transfer Learning no FF++', fontsize=13)
# plt.xlabel('Época')
# plt.ylabel('Acurácia')
# plt.legend(loc='lower right')
# plt.ylim(0, 1)
# plt.tight_layout()
# plt.show()



# auc_all     = history.history['auc']     + history2.history['auc']
# val_auc_all = history.history['val_auc'] + history2.history['val_auc']

# plt.figure(figsize=(10, 5))
# plt.plot(auc_all,     label='Treino',     linewidth=2)
# plt.plot(val_auc_all, label='Validação',  linewidth=2)
# plt.axvline(x=EPOCHS - 0.5, color='gray', linestyle='--', linewidth=1.5,
#             label='Início do fine-tuning')
# plt.fill_betweenx([0, 1], 0, EPOCHS - 0.5,
#                   alpha=0.05, color='blue',  label='Extração de características')
# plt.fill_betweenx([0, 1], EPOCHS - 0.5, len(auc_all),
#                   alpha=0.05, color='orange', label='Fine-tuning')
# plt.title('AUC — Transfer Learning no FF++', fontsize=13)
# plt.xlabel('Época')
# plt.ylabel('AUC')
# plt.legend(loc='lower right')
# plt.ylim(0, 1)
# plt.tight_layout()
# plt.show()